# MuduoXinyu Attention 实验

按顺序运行下面的代码，需要修改的参数已在代码中标出。


<a id="s-02-02_multi_operator_vs_flash_attention"></a>


## 9.6 多算子调用与 FlashAttention

本节从下载源码和模型开始，依次完成编译和 A/B 比较。请在本 Notebook 所在目录执行，使用下面给出的默认路径。

源码来源和课程版本见 [src/UPSTREAM.md](src/UPSTREAM.md)。


### Attention 在算什么

每个 token 先得到 Query（查询）、Key（索引）和 Value（内容）。计算 `softmax(QKᵀ / √d) V`：点积判断关联程度，除以 `√d` 控制数值尺度，softmax 把分数变成权重，再对 Value 加权求和。因果掩码让当前 token 只能看到自己和之前的位置。

逐 token 生成时，KV cache 保存先前 token 的 Key 和 Value，避免每一步重复计算。序列越长，当前 Query 要读取的历史 KV 越多。融合实现把多个计算步骤合并，减少中间数据读写和算子提交次数；小形状下，这些收益也可能被准备和精度转换开销抵消。

本实验 Path A 使用 FP32 多算子，Path B 使用 FP16 FlashAttentionV4，并包含必要的类型转换。因此比较的是这两套完整实现。先检查生成的 token 是否一致，再比较吞吐量。

### 准备源码和模型

源码来自 BNU-SYS，下面下载公开教学副本的 `course-attention-ab` 版本，已包含 A/B 两条实现。模型使用 stories110M（约 419 MiB）及配套 tokenizer；`smoke_tiny.bin` 不适用于本实验。

已有源码或模型时跳过对应下载格，在路径设置格填写位置。离线环境需提前准备好源码、模型和 tokenizer。编译完成后，重做 A/B 可直接运行第 3、4 步；需要保留前一次结果时，再更换 `OUTPUT_DIR`。


In [ ]:
%%bash
set -euo pipefail
git clone --branch course-attention-ab https://gitcode.com/bnujwx/muduoXinyu.git muduoxinyu-demo
mkdir -p assets results


下面通过 Hugging Face 镜像下载模型，tokenizer 从原项目下载。已有文件时跳过这一格，直接设置路径；下载失败后只需重跑这一格。


In [ ]:
%%bash
set -euo pipefail
mkdir -p assets
curl -fL --retry 3 --connect-timeout 20 https://hf-mirror.com/karpathy/tinyllamas/resolve/main/stories110M.bin -o assets/stories110M.bin
curl -fL --retry 3 --connect-timeout 20 https://raw.githubusercontent.com/karpathy/llama2.c/c02865df300f3bd9e567ce061000dc23bf785a17/tokenizer.bin -o assets/tokenizer.bin


In [ ]:
# 文件已放在其他位置时修改路径。
%env MUDUO_ROOT=muduoxinyu-demo
%env MODEL=assets/stories110M.bin
%env TOKENIZER=assets/tokenizer.bin
%env OUTPUT_DIR=results/attention_ab


### 1. 查看设备

确认 NPU 健康状态为 OK。


In [ ]:
%%bash
npu-smi info


### 2. 构建 NPU 版本

编译日志保存在 `results/build.log`，出错时可打开查看。已编译好的程序可以直接继续运行。


In [ ]:
%%bash
set -eo pipefail
mkdir -p results
make -C "$MUDUO_ROOT" -j8 npu ASCEND_PATH="$ASCEND_HOME_PATH" 2>&1 | tee results/build.log


### 3. 运行两种实现

下面的代码使用前面准备的模型、tokenizer 和源码，调用脚本完成比较：

- A：FP32 多算子 Attention。
- B：FP16 FlashAttentionV4，包含必要的类型转换。

两种实现使用相同的提示词、生成长度、解码参数和设备。脚本先按 A→B 检查功能，再按 B→A 测量性能；每种实现预热 1 次，统计后面 3 次。日志和结果保存在前面设置的 `OUTPUT_DIR` 目录。

重复比较也从本步骤继续。确认程序、模型和 tokenizer 已准备好后，直接运行下面的单元格。

| 条件 | 脚本默认值 |
|---|---|
| 提示词 | hello |
| 短功能检查 | 每条路径 12 步 |
| 性能测试 | 每条路径 120 步；四个相同提示词样本 |
| 解码 | temperature=0，topP=1 |
| 统计 | 第一个样本预热，后三个求均值和 CV |

CV 是标准差除以均值的百分比，表示这几个样本的波动。改变生成长度时，修改 `run_ab_benchmark.sh` 开头的 `PERF_STEPS`，再重跑两种实现。


In [ ]:
%%bash
bash src/run_ab_benchmark.sh \
  --muduo-root "$MUDUO_ROOT" \
  --model "$MODEL" --tokenizer "$TOKENIZER" \
  --output-dir "$OUTPUT_DIR"


### 4. 读取结论


In [ ]:
import json
import os
from pathlib import Path

result = json.loads((Path(os.environ["OUTPUT_DIR"]) / "result.json").read_text())
print("功能检查通过：", result["verdict"]["functional_pass"])
print("性能数据有效：", result["verdict"]["performance_valid"])
print("本批 B 均值更高：", result["verdict"]["performance_beneficial"])
for path in ("path_a", "path_b"):
    perf = result["perf"][path]
    print(path, f"{perf['mean_tokens_per_s']:.2f} tokens/s，波动 {perf['cv_percent']:.2f}%")
print(f"B 相对 A 的吞吐量变化：{result['perf']['delta_percent_b_vs_a']:.2f}%")

print("短检查调用次数：", result["smoke"]["flash_stats"])
print("短检查结果：", result["smoke"]["checks"])


`functional_pass` 检查 token、执行路径与 fallback；`performance_valid` 检查样本和工作量；`performance_beneficial` 只表示本批 B 的均值更高。

短检查使用 12 层、12 步，所以期望 144 次 FlashAttention 和 576 次 Cast。这里的计数来自短功能检查，不是 120 步性能批次的总次数。性能批次也会检查生成 token 是否一致。

本次 A 为 163.54 tokens/s，B 为 165.41 tokens/s，B 高约 1.14%；A、B 的波动分别为 0.39% 和 1.79%。提升幅度与波动接近，可以说明本次 B 的均值更高，还需重复测量才能判断是否有稳定收益。

### 课后实践

调用次数减少为什么仍可能变慢？结合类型转换、布局、启动成本和波动解释，再设计一个只改变一项条件的实验。参考答案见 `answer/09.06_answer.md`。
